# Notebook 03: Feature Engineering and Transformation
**Mục tiêu:**
1. Chia dữ liệu theo trục thời gian (Train/Val/Test) khắt khe để chống rò rỉ dữ liệu.
2. Gọi module Feature Engineering để tính Log Returns, Lạm phát và Lags.
3. Chạy kiểm định tính dừng (Stationarity) cho các biến đưa vào mô hình.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Cấu hình hiển thị
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Cấu hình đường dẫn
ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = ROOT / "data" / "processed"
SPLITS_DIR = PROCESSED_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# THÊM THƯ MỤC SRC VÀO ĐƯỜNG DẪN ĐỂ IMPORT MODULES
sys.path.append(str(ROOT / "src"))
from feature_engineering import chronological_split, apply_fe_pipeline
from statistical_tests import run_adf_test

print("Đã load thành công các custom modules!")

Đã load thành công các custom modules!


In [2]:
# Load dữ liệu sạch từ Notebook 02
master_df = pd.read_csv(PROCESSED_DIR / "master_data_monthly.csv", index_col='DATE', parse_dates=True)
master_df = master_df.sort_index()

print(f"Toàn bộ dữ liệu: {master_df.index.min().date()} đến {master_df.index.max().date()}")

Toàn bộ dữ liệu: 1986-01-31 đến 2026-04-30


## 1. Phân chia Train/Val/Test (Chronological Split)
Chúng ta cắt dữ liệu nguyên thủy thành 3 phần (70% - 15% - 15%) **trước khi** thực hiện bất kỳ phép toán nào để đảm bảo mô hình không học lén tương lai.

In [3]:
# Gọi hàm từ module tự viết
train_raw, val_raw, test_raw = chronological_split(master_df, train_ratio=0.70, val_ratio=0.15)

print(f"Tập Train: {len(train_raw)} tháng ({train_raw.index.min().date()} -> {train_raw.index.max().date()})")
print(f"Tập Val:   {len(val_raw)} tháng ({val_raw.index.min().date()} -> {val_raw.index.max().date()})")
print(f"Tập Test:  {len(test_raw)} tháng ({test_raw.index.min().date()} -> {test_raw.index.max().date()})")

Tập Train: 338 tháng (1986-01-31 -> 2014-02-28)
Tập Val:   73 tháng (2014-03-31 -> 2020-03-31)
Tập Test:  73 tháng (2020-04-30 -> 2026-04-30)


## 2. Feature Engineering với cơ chế Look-back
Nếu tập Validation chỉ tự tính sai phân, nó sẽ bị mất dòng đầu tiên thành `NaN`. Hàm `apply_fe_pipeline` của chúng ta sẽ nối mượn 3 dòng cuối của tập trước đó (lookback), tính toán xong rồi lại cắt đi, giúp bảo toàn 100% độ dài của tập Val và Test.

In [4]:
MAX_LAG = 3

# 1. Train tự xử lý (sẽ mất MAX_LAG dòng ở đầu do shift)
train_fe = apply_fe_pipeline(train_raw, lookback_df=None, max_lag=MAX_LAG)

# 2. Val mượn đuôi của Train (Bảo toàn 100% dòng)
val_fe = apply_fe_pipeline(val_raw, lookback_df=train_raw, max_lag=MAX_LAG)

# 3. Test mượn đuôi của Val (Bảo toàn 100% dòng)
test_fe = apply_fe_pipeline(test_raw, lookback_df=val_raw, max_lag=MAX_LAG)

print(f"Số dòng Train sau FE: {len(train_fe)} (bị drop {MAX_LAG} dòng khởi tạo)")
print(f"Số dòng Val sau FE:   {len(val_fe)} (giữ nguyên gốc: {len(val_raw)})")
print(f"Số dòng Test sau FE:  {len(test_fe)} (giữ nguyên gốc: {len(test_raw)})")

display(train_fe[['NDX_log_ret', 'CPI_inflation', 'FFR_diff']].head())

Số dòng Train sau FE: 334 (bị drop 3 dòng khởi tạo)
Số dòng Val sau FE:   73 (giữ nguyên gốc: 73)
Số dòng Test sau FE:  73 (giữ nguyên gốc: 73)


,NDX_log_ret,CPI_inflation,FFR_diff
DATE,,,
1986-05-31,0.051887,0.002756,-0.14
1986-06-30,-0.003315,0.003663,0.07
1986-07-31,-0.117998,0.000914,-0.36
1986-08-31,0.048487,0.000913,-0.39
1986-09-30,-0.098281,0.003643,-0.28


## 3. Kiểm định Augmented Dickey-Fuller (ADF)
Mô hình SVAR và SARIMA (phần AR/MA) yêu cầu các chuỗi dữ liệu phải dừng $I(0)$. Chúng ta chạy kiểm định ADF cho các biến vừa được biến đổi trên tập **Train** (không chạy trên tập Test để khách quan).

In [6]:
# Các biến mục tiêu sẽ đưa vào mô hình vĩ mô
target_cols = ['NDX_log_ret', 'CPI_inflation', 'FFR_diff']

adf_results = []
for col in target_cols:
    res = run_adf_test(train_fe[col])
    res['Variable'] = col
    adf_results.append(res)

# Tạo DataFrame kết quả đẹp mắt
adf_df = pd.DataFrame(adf_results)
adf_df = adf_df[['Variable', 'ADF_Statistic', 'p_value', 'Num_Lags', 'Is_Stationary']]

print("KẾT QUẢ KIỂM ĐỊNH TÍNH DỪNG (ADF TEST):")
display(adf_df)


KẾT QUẢ KIỂM ĐỊNH TÍNH DỪNG (ADF TEST):


,Variable,ADF_Statistic,p_value,Num_Lags,Is_Stationary
0,NDX_log_ret,-16.8034,0.0000,0,True
1,CPI_inflation,-3.6842,0.0043,14,True
2,FFR_diff,-4.0652,0.0011,10,True


## 4. Xuất Dữ Liệu
Lưu 3 file CSV cuối cùng. Đây sẽ là nguyên liệu tiêu chuẩn, đóng băng (frozen) để cấp cho Notebook 04 (SVAR) và Notebook 05, 06 (SARIMA).

In [7]:
# Lưu ra thư mục processed/splits/
train_fe.to_csv(SPLITS_DIR / "train_fe.csv")
val_fe.to_csv(SPLITS_DIR / "val_fe.csv")
test_fe.to_csv(SPLITS_DIR / "test_fe.csv")

print(f"Đã xuất thành công 3 file sẵn sàng cho Mô hình hóa vào thư mục: {SPLITS_DIR}")

Đã xuất thành công 3 file sẵn sàng cho Mô hình hóa vào thư mục: S:\vscode\nasdaq-macro-shocks-analysis\data\processed\splits
